# HomeAI · Medición de Qwen-Image-Edit en Colab (GPU gratuita)

Ejecuta el pipeline de generación de HomeAI **con el mismo código que el servidor**
(instrucción de edición → 2 variantes → filtro de similitud antes del puntuador →
regeneración de las casi idénticas → puntuador estético → la mejor), pero con el
modelo en la GPU de Colab en lugar de la Inference API de pago. No gasta crédito
de Hugging Face y **no necesita ningún token**.

**Cómo usarlo**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU T4*** → Guardar.
2. *Entorno de ejecución → **Ejecutar todo***. No hay que tocar nada más.
3. Al final se descarga `qa_qwen.zip` (informe, datos y rejillas). Súbelo a
   `server/validation/results_colab/` del repositorio o pásaselo a Claude.

**Qué hace y cuánto tarda (estimación, se mide al ejecutarlo)**
- Descarga ~58 GB de pesos (codificador Qwen2.5-VL 7B + transformer de 20B) y los
  carga **en 4 bits** para que quepan en los 16 GB de la T4, en dos fases (primero
  codifica todas las instrucciones, luego genera), borrando de disco lo que ya usó.
- 10 fotos reales × 2 variantes = 20 generaciones (+ las regeneradas), con la
  LoRA *Lightning* (8 pasos). Después, SDXL sobre las mismas fotos como referencia.
- Si Colab se desconecta, vuelve a *Ejecutar todo*: empieza de cero.
- ¿Se queda sin memoria o sale todo negro? Cambia `QWEN_DTYPE` en la celda de
  configuración y vuelve a ejecutar; el informe dice qué falló.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "diffusers>=0.35" "transformers>=4.51" "accelerate>=1.0" "bitsandbytes>=0.46" "peft>=0.17"
import shutil
print(f"Disco libre: {shutil.disk_usage('/content').free / 2**30:.0f} GB (hacen falta ~45 GB a la vez)")

## Código del servidor de HomeAI (generado desde el repositorio, no editar aquí)

In [ ]:
%%writefile pipeline.py
"""Lógica del pipeline de generación que no depende de FastAPI.

La usan el servidor (app.py) y el notebook de Colab
(validation/colab/build_notebook.py lo incrusta tal cual), así que la medición
de QA ejecuta exactamente el mismo código que producción:

    prompt -> N variantes -> filtro de similitud (casi idénticas al original:
    descartar y regenerar una vez) -> puntuador estético solo sobre las que
    pasan -> umbral de calidad -> la mejor

Modelo: Qwen/Qwen-Image-Edit (Apache-2.0). Es un modelo de EDICIÓN por
instrucciones, no un img2img tipo SDXL: el texto lo lee Qwen2.5-VL viendo la
foto (semántica) y el VAE aporta la apariencia de la foto original. Por eso
el prompt es una instrucción ("Redecora esta habitación...") y no una lista
de etiquetas, y no existe "strength": la intensidad del cambio se pide en la
propia instrucción.
"""

from __future__ import annotations

import numpy as np

import segmentation as seg

# --- Prompts -----------------------------------------------------------------

# Modelos img2img clásicos (SDXL y similares): prompt descriptivo + strength.
STRUCTURE_SUFFIX = (
    "same room, same camera angle, keep walls, windows, doors, ceiling and floor layout "
    "exactly in place, realistic interior design photograph, natural light, high detail"
)
DEFAULT_NEGATIVE = (
    "different room, changed layout, moved windows, extra doors, distorted walls, "
    "warped perspective, people, text, watermark, blurry, low quality, cartoon"
)

# Modelos de edición por instrucciones (Qwen-Image-Edit, FLUX Kontext...).
EDIT_KEEP = (
    "Keep the same room and the same camera angle: walls, windows, doors, ceiling height and floor layout "
    "stay exactly where they are. The result must look like a real, photorealistic interior photograph "
    "with natural light."
)
# strength (0,30-0,80 en la interfaz) -> cuánto se pide cambiar.
EDIT_INTENSITY = [
    (0.45, "Make a subtle update: change only colours, textiles, lighting and decoration, keep the existing furniture."),
    (0.65, "Replace the furniture, finishes and decoration to match this style."),
    (1.01, "Do a complete redesign of furniture, finishes, lighting and decoration in this style."),
]


def is_edit_model(model: str) -> bool:
    m = model.lower()
    return any(k in m for k in ("image-edit", "kontext", "instruct")) or (m.startswith("http") and "edit" in m)


def build_prompt(style: str, zones: list[str], strength: float = 0.55, edit: bool = False) -> str:
    if not edit:
        if not zones:
            return f"{style}, {STRUCTURE_SUFFIX}"
        target = " and ".join(seg.ZONE_PROMPT[z] for z in zones)
        return f"{target} redesigned: {style}, {STRUCTURE_SUFFIX}"
    intensity = next(text for limit, text in EDIT_INTENSITY if strength < limit)
    if zones:
        target = " and ".join(seg.ZONE_PROMPT[z] for z in zones)
        head = f"Redesign only {target} of this room in this style: {style}. Leave everything else unchanged."
    else:
        head = f"Redecorate this room in this style: {style}."
    return f"{head} {intensity} {EDIT_KEEP}"


# --- Filtro de similitud, puntuación y elección ------------------------------

def zone_box(mask_arr, size, pad: int = 8):
    """Caja de la zona elegida (+pad) donde se mide la similitud."""
    if mask_arr is None:
        return None
    ys, xs = np.nonzero(mask_arr)
    w, h = size
    return (max(0, int(xs.min()) - pad), max(0, int(ys.min()) - pad),
            min(w, int(xs.max()) + pad + 1), min(h, int(ys.max()) + pad + 1))


def check_similarity(variants: list[dict], original, box, max_similarity: float) -> bool:
    """Paso 1, ANTES del puntuador: marca como descartadas («sin_cambios») las
    variantes casi idénticas al original. Devuelve False si no se pudo medir
    (sin CLIP): entonces no se descarta nada."""
    import quality

    try:
        for v in variants:
            v["similarity"] = quality.similarity(v["image"], original, box)
    except Exception:  # noqa: BLE001 - sin modelo se sigue sin filtrar
        return False
    for v in variants:
        near = v["similarity"] > max_similarity
        v["discarded"], v["reason"] = near, ("sin_cambios" if near else None)
    return True


def score_variants(variants: list[dict], min_score: float) -> bool:
    """Paso 2: puntuador estético SOLO sobre las que pasaron el filtro de
    similitud; las de nota baja se marcan «calidad_baja». Devuelve False si el
    puntuador falla (se devuelven sin filtrar por calidad)."""
    import quality

    kept = [v for v in variants if not v.get("discarded")]
    try:
        for v in kept:
            v.update(quality.score(v["image"]))
    except Exception:  # noqa: BLE001
        return False
    for v in kept:
        if v["score"] < min_score:
            v["discarded"], v["reason"] = True, "calidad_baja"
    return True


def pick_best(variants: list[dict]) -> int:
    """La de más nota entre las no descartadas. Si todas lo están, antes una de
    «calidad baja» que una «sin cambios» (mostrar la foto tal cual no sirve), y
    entre las «sin cambios», la que menos se parece al original."""

    def key(i):
        v = variants[i]
        if v.get("reason") == "sin_cambios":
            return (2, v.get("similarity") if v.get("similarity") is not None else 1.0)
        return (1 if v.get("reason") else 0, -(v.get("score") or 0))

    return min(range(len(variants)), key=key)


In [ ]:
%%writefile segmentation.py
"""Segmentación por zonas y composición con máscara.

- segment(): SegFormer (ADE20K) ejecutado dentro del servicio. Se eligió b4
  tras validarlo con 10 fotos reales frente a b0/b2 (ver
  server/validation/results_compare/compare.md y CURRENT_STATE.md).
- zone_map(): agrupa las 150 clases de ADE20K en las zonas de HomeAI.
- composite(): pega el resultado del modelo SOLO dentro de la máscara. El
  borde se suaviza hacia dentro, nunca hacia fuera: fuera de la máscara la
  foto queda idéntica píxel a píxel, sirva el proveedor inpainting o no.

torch/transformers se importan solo al cargar el modelo, para que el resto
del servicio (y los tests) no dependan de ellos.
"""

from __future__ import annotations

import os
import threading

import numpy as np
from PIL import Image, ImageFilter

SEG_MODEL = os.environ.get("SEG_MODEL", "nvidia/segformer-b4-finetuned-ade-512-512")

# Orden fijo: el índice es el valor que se envía al navegador en el mapa.
ZONE_ORDER = ["pared", "suelo", "techo", "ventana/puerta", "mobiliario", "otros"]
SELECTABLE = ["pared", "suelo", "techo", "mobiliario"]
ZONE_PROMPT = {
    "pared": "the walls",
    "suelo": "the floor",
    "techo": "the ceiling",
    "ventana/puerta": "the windows and doors",
    "mobiliario": "the furniture",
    "otros": "the selected area",
}
ZONES = {
    "pared": {"wall", "column"},
    "suelo": {"floor"},
    "techo": {"ceiling"},
    "ventana/puerta": {"windowpane", "window", "door", "screen door", "double door"},
    # "mirror" queda fuera a propósito: refleja paredes y techo.
    "mobiliario": {
        "bed", "cabinet", "table", "chair", "sofa", "shelf", "armchair", "seat", "desk",
        "wardrobe", "lamp", "bathtub", "cushion", "chest of drawers", "counter", "sink",
        "fireplace", "refrigerator", "pillow", "bookcase", "coffee table", "toilet",
        "countertop", "stove", "kitchen island", "swivel chair", "chandelier", "ottoman",
        "buffet", "stool", "oven", "microwave", "dishwasher", "shower", "radiator",
        "television receiver", "curtain", "blind", "rug", "painting", "plant",
        "bench", "cradle", "sconce", "hood", "washer", "towel", "vase", "clock", "light",
    },
}
# El mapa se envía como PNG en gris con este paso entre zonas, para que una
# pequeña deriva de color al decodificarlo en el navegador no cambie la zona.
MAP_STEP = 40


def norm(label: str) -> str:
    return label.strip().lower().split(",")[0].strip()


def zone_of(label: str) -> str:
    n = norm(label)
    for zone, names in ZONES.items():
        if n in names:
            return zone
    return "otros"


def zone_map(labels: np.ndarray, id2label: dict) -> np.ndarray:
    lut = np.array([ZONE_ORDER.index(zone_of(id2label[i])) for i in range(len(id2label))], dtype=np.uint8)
    return lut[labels]


class _Segmenter:
    def __init__(self, model_id: str):
        import torch
        from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

        torch.set_num_threads(max(1, os.cpu_count() or 1))
        self.torch = torch
        self.proc = SegformerImageProcessor.from_pretrained(model_id)
        self.model = SegformerForSemanticSegmentation.from_pretrained(model_id).eval()
        self.id2label = self.model.config.id2label

    def __call__(self, img: Image.Image) -> np.ndarray:
        torch = self.torch
        with torch.no_grad():
            logits = self.model(**self.proc(images=img, return_tensors="pt")).logits
            logits = torch.nn.functional.interpolate(logits, size=img.size[::-1], mode="bilinear", align_corners=False)
            labels = logits.argmax(dim=1)[0].numpy()
        return zone_map(labels, self.id2label)


_segmenter = None
_segmenter_lock = threading.Lock()


def load_segmenter():
    """Carga perezosa y única del modelo (tarda unos segundos la primera vez)."""
    global _segmenter
    with _segmenter_lock:
        if _segmenter is None:
            _segmenter = _Segmenter(SEG_MODEL)
    return _segmenter


def segment(img: Image.Image) -> np.ndarray:
    """Devuelve un mapa HxW de índices de ZONE_ORDER, suavizado para quitar
    motas sueltas (un filtro de moda de 5 px)."""
    zmap = load_segmenter()(img)
    smooth = Image.fromarray(zmap.astype(np.uint8)).filter(ImageFilter.ModeFilter(5))
    return np.asarray(smooth, dtype=np.uint8)


def zones_summary(zmap: np.ndarray) -> list[dict]:
    total = zmap.size
    return [
        {"id": i, "zone": z, "pct": round(100 * float((zmap == i).sum()) / total, 1), "selectable": z in SELECTABLE}
        for i, z in enumerate(ZONE_ORDER)
    ]


def map_to_png(zmap: np.ndarray) -> Image.Image:
    return Image.fromarray((zmap.astype(np.uint16) * MAP_STEP).astype(np.uint8), mode="L")


def parse_mask(img: Image.Image, size: tuple[int, int]) -> np.ndarray:
    """Máscara del navegador (blanco = modificar) -> booleano al tamaño de la foto."""
    mask = img.convert("L")
    if mask.size != size:
        mask = mask.resize(size, Image.NEAREST)
    return np.asarray(mask) > 127


def composite(original: Image.Image, generated: Image.Image, mask: np.ndarray) -> Image.Image:
    """Resultado del modelo dentro de la máscara, original fuera.

    El alfa es un desenfoque de la máscara recortado por la propia máscara:
    transición suave hacia dentro del borde y exactamente 0 fuera de él.
    """
    if generated.size != original.size:
        generated = generated.resize(original.size, Image.LANCZOS)
    radius = max(2, round(max(original.size) * 0.006))
    hard = Image.fromarray((mask * 255).astype(np.uint8), mode="L")
    soft = np.asarray(hard.filter(ImageFilter.GaussianBlur(radius)), dtype=np.float32) / 255.0
    alpha = np.where(mask, np.clip((soft - 0.5) * 2, 0, 1), 0.0)
    a = alpha[..., None]
    out = np.asarray(generated.convert("RGB"), dtype=np.float32) * a + np.asarray(original, dtype=np.float32) * (1 - a)
    result = Image.fromarray(np.clip(out + 0.5, 0, 255).astype(np.uint8))
    # Garantía explícita: fuera de la máscara, los píxeles originales tal cual.
    result.paste(original, mask=Image.fromarray(((~mask) * 255).astype(np.uint8), mode="L"))
    return result



In [ ]:
%%writefile style.py
"""Detección automática del estilo actual de la habitación (CLIP zero-shot).

Elegido tras validarlo con fotos reales (server/validation/validate_style.py,
results_style/report.md y CURRENT_STATE.md):
- openai/clip-vit-large-patch14: 6/7 aciertos en fotos de estilo claro frente
  a 3-4/8 de base/32.
- Solo se sugiere si la probabilidad del primero es >= 0,5 y le saca >= 0,2
  al segundo: con eso, las sugerencias mostradas acertaron 6/6.
- Etiquetas "sumidero" (habitación vacía, primer plano de un objeto, exterior):
  si ganan, no se sugiere nada. Sin ellas, una habitación vacía salía
  "minimalista" con 0,97.
- Sin validar con fotos buenas: escandinavo, bohemio y mediterráneo.

torch/transformers se importan solo al cargar el modelo.
"""

from __future__ import annotations

import os
import threading

from PIL import Image

STYLE_MODEL = os.environ.get("STYLE_MODEL", "openai/clip-vit-large-patch14")
MIN_PROB = float(os.environ.get("STYLE_MIN_PROB", 0.5))
MIN_MARGIN = float(os.environ.get("STYLE_MIN_MARGIN", 0.2))

# clave -> (nombre para mostrar, texto para CLIP). Mismas frases que en la validación.
STYLES = {
    "moderno": ("Moderno", "a modern contemporary interior"),
    "rustico": ("Rústico", "a rustic interior with exposed wood beams and stone"),
    "minimalista": ("Minimalista", "a minimalist interior with very few objects"),
    "industrial": ("Industrial", "an industrial loft interior with exposed brick, concrete and metal"),
    "escandinavo": ("Escandinavo", "a scandinavian interior with white walls and light wood"),
    "clasico": ("Clásico", "a classic traditional interior with ornate furniture and moldings"),
    "bohemio": ("Bohemio", "a bohemian interior with plants, rugs and colorful patterns"),
    "mediterraneo": ("Mediterráneo", "a mediterranean interior with whitewashed walls and terracotta"),
}
SINKS = {
    "vacia": "an empty unfurnished room with bare walls",
    "objeto": "a close-up photo of a single object or piece of furniture",
    "exterior": "the outside of a building seen from the street or garden",
}
SINK_REASON = {
    "vacia": "La habitación parece vacía: no hay muebles que indiquen un estilo.",
    "objeto": "La foto parece un detalle de cerca, no la estancia entera.",
    "exterior": "La foto parece el exterior de un edificio.",
}
KEYS = list(STYLES) + list(SINKS)
TEMPLATE = "a photo of {}"


class _Clip:
    def __init__(self, model_id: str):
        import torch
        from transformers import CLIPModel, CLIPProcessor

        torch.set_num_threads(max(1, os.cpu_count() or 1))
        self.torch = torch
        self.model = CLIPModel.from_pretrained(model_id).eval()
        self.proc = CLIPProcessor.from_pretrained(model_id)
        self.texts = [TEMPLATE.format(t) for _, t in STYLES.values()] + [TEMPLATE.format(t) for t in SINKS.values()]

    def __call__(self, img: Image.Image) -> list[float]:
        # logits_per_image: salida estable entre versiones de transformers.
        with self.torch.no_grad():
            inputs = self.proc(text=self.texts, images=img, return_tensors="pt", padding=True)
            return self.model(**inputs).logits_per_image.softmax(dim=-1)[0].tolist()


_clip = None
_clip_lock = threading.Lock()


def load_classifier():
    global _clip
    with _clip_lock:
        if _clip is None:
            _clip = _Clip(STYLE_MODEL)
    return _clip


def decide(probs: list[float]) -> dict:
    """Convierte las probabilidades en la respuesta de la API.

    suggest=False cuando gana una etiqueta sumidero o el modelo duda (poca
    probabilidad o poca ventaja sobre el segundo). En ese caso la interfaz no
    muestra ningún estilo como detectado.
    """
    ranked = sorted(zip(KEYS, probs), key=lambda kv: -kv[1])
    (top, p1), (_, p2) = ranked[0], ranked[1]
    styles_only = [(k, p) for k, p in ranked if k in STYLES][:3]
    result = {
        "style": None,
        "label": None,
        "confidence": round(p1, 3),
        "margin": round(p1 - p2, 3),
        "suggest": False,
        "reason": None,
        "top": [{"style": k, "label": STYLES[k][0], "p": round(p, 3)} for k, p in styles_only],
    }
    if top in SINKS:
        result["reason"] = SINK_REASON[top]
    elif p1 < MIN_PROB or p1 - p2 < MIN_MARGIN:
        result["reason"] = "No hay un estilo claro en la foto (o mezcla varios)."
    else:
        result.update(style=top, label=STYLES[top][0], suggest=True)
    return result


def classify(img: Image.Image) -> dict:
    return decide(load_classifier()(img))


In [ ]:
%%writefile quality.py
"""Puntuación de calidad de las imágenes generadas (para elegir la mejor variante).

Dos señales sobre el mismo CLIP large/14 que ya usa la detección de estilo
(style.py), así que no se carga otro modelo grande:

- aesthetic: predictor estético de LAION ("improved-aesthetic-predictor",
  Christoph Schuhmann, Apache-2.0): una MLP pequeña sobre el embedding de
  imagen de CLIP ViT-L/14, entrenada con valoraciones humanas (escala ~1-10).
- realism: CLIP zero-shot, probabilidad de "foto nítida y realista de una
  habitación" frente a "imagen distorsionada, borrosa o con artefactos" (0-1).

score = aesthetic normalizada (0-1) * 0,5 + realism * 0,5. Umbrales de
descarte calibrados en server/validation/check_quality.py.

Aparte, similarity(): similitud coseno entre los embeddings CLIP de la
variante y del original. Se calcula ANTES de puntuar (app.py) para descartar o
regenerar variantes en las que el modelo "no hizo nada" sin gastar el
puntuador en ellas. La diferencia de píxeles o un hash perceptual no sirven:
una variante solo más luminosa movía 44 niveles de media, casi como una
reforma real, y CLIP sí la reconoce como la misma imagen.
"""

from __future__ import annotations

import os
import threading

from PIL import Image

import style

AESTHETIC_URL = os.environ.get(
    "AESTHETIC_WEIGHTS_URL",
    "https://github.com/christophschuhmann/improved-aesthetic-predictor/raw/main/sac+logos+ava1-l14-linearMSE.pth",
)
GOOD = "a sharp, realistic, well lit photo of a room interior"
BAD = ["a distorted, warped image with visual artifacts", "a blurry, noisy, low quality image",
       "an image with melted furniture and impossible geometry"]


class _Scorer:
    def __init__(self):
        import io

        import requests
        import torch

        self.torch = torch
        clip = style.load_classifier()  # reutiliza CLIP large/14 ya cargado
        self.model, self.proc = clip.model, clip.proc
        layers = [torch.nn.Linear(768, 1024), torch.nn.Dropout(0.2), torch.nn.Linear(1024, 128), torch.nn.Dropout(0.2),
                  torch.nn.Linear(128, 64), torch.nn.Dropout(0.1), torch.nn.Linear(64, 16), torch.nn.Linear(16, 1)]
        self.mlp = torch.nn.Sequential(*layers)
        raw = requests.get(AESTHETIC_URL, timeout=60).content
        state = torch.load(io.BytesIO(raw), map_location="cpu", weights_only=True)  # solo tensores, sin pickle de código
        self.mlp.load_state_dict({k.replace("layers.", ""): v for k, v in state.items()})
        self.mlp.eval()
        self.texts = [GOOD] + BAD

    def embed(self, img: Image.Image):
        # Embedding de imagen por las capas explícitas: estable entre versiones
        # de transformers (get_image_features cambió de tipo de retorno).
        torch = self.torch
        with torch.no_grad():
            pix = self.proc(images=img.convert("RGB"), return_tensors="pt")["pixel_values"]
            pooled = self.model.vision_model(pixel_values=pix).pooler_output
            return torch.nn.functional.normalize(self.model.visual_projection(pooled), dim=-1)

    def similarity(self, img: Image.Image, original: Image.Image, box=None) -> float:
        if box is not None:
            # Con zonas: se compara el recorte de la zona elegida, no la imagen
            # entera (si solo cambia el suelo, la imagen entera apenas "cambia").
            img, original = img.crop(box), original.crop(box)
        return float((self.embed(img) @ self.embed(original).T)[0, 0])

    def __call__(self, img: Image.Image) -> dict:
        torch = self.torch
        with torch.no_grad():
            aesthetic = float(self.mlp(self.embed(img))[0, 0])
            inputs = self.proc(text=self.texts, images=img.convert("RGB"), return_tensors="pt", padding=True)
            realism = float(self.model(**inputs).logits_per_image.softmax(dim=-1)[0][0])
        a_norm = min(1.0, max(0.0, (aesthetic - 3.0) / 4.0))  # ~3 → 0, ~7 → 1
        return {"aesthetic": round(aesthetic, 3), "realism": round(realism, 3), "score": round(0.5 * a_norm + 0.5 * realism, 3)}


_scorer = None
_lock = threading.Lock()


def load_scorer():
    global _scorer
    with _lock:
        if _scorer is None:
            _scorer = _Scorer()
    return _scorer


def score(img: Image.Image) -> dict:
    return load_scorer()(img)


def similarity(img: Image.Image, original: Image.Image, box=None) -> float:
    """1,0 = misma imagen para CLIP. Antes era change = 1 - similarity."""
    return round(load_scorer().similarity(img, original, box), 4)


In [ ]:
%%writefile local_qwen.py
"""Qwen/Qwen-Image-Edit ejecutado con diffusers (QwenImageEditPipeline).

Dos usos, mismo código:

1. Servidor con GPU propia: GEN_BACKEND=diffusers en app.py. get_client()
   devuelve un objeto con la misma interfaz que InferenceClient.image_to_image,
   así que el resto del pipeline (filtro de similitud, puntuador) no cambia.
   Requiere requirements-gpu.txt y una GPU de 24 GB+ con QWEN_QUANT=nf4
   (o ~60 GB sin cuantizar).
2. Notebook de Colab (validation/colab/): GPU T4 gratuita de 16 GB. Ahí no
   caben a la vez el codificador de texto (Qwen2.5-VL 7B) y el transformer
   (20B), ni siquiera en 4 bits, así que se hace en dos fases: primero se
   codifican todas las instrucciones (load(parts="text") + encode()), se
   libera la GPU, y luego se generan las imágenes (load(parts="image") +
   render()).

Modelo de edición por instrucciones: el texto lo lee Qwen2.5-VL VIENDO la foto
y el VAE aporta la apariencia original, por eso la instrucción se codifica
junto con la imagen (encode recibe las dos) y no hay parámetro "strength".
La guía es true_cfg_scale (con prompt negativo); guidance_scale no se usa.

Opcional: LoRA "Lightning" (lightx2v, Apache-2.0) para generar en 8 pasos sin
CFG en vez de 30-50 pasos con CFG (unas 8-12 veces menos cómputo).
"""

from __future__ import annotations

import io
import math
import os
import random
import threading

from PIL import Image

MODEL = "Qwen/Qwen-Image-Edit"
LIGHTNING_REPO = "lightx2v/Qwen-Image-Lightning"
LIGHTNING_FILE = os.environ.get("QWEN_LIGHTNING_FILE", "Qwen-Image-Edit-Lightning-8steps-V1.0-bf16.safetensors")
# Configuración del planificador que publica lightx2v para sus LoRA.
LIGHTNING_SCHEDULER = {
    "base_image_seq_len": 256,
    "base_shift": math.log(3),
    "invert_sigmas": False,
    "max_image_seq_len": 8192,
    "max_shift": math.log(3),
    "num_train_timesteps": 1000,
    "shift": 1.0,
    "shift_terminal": None,
    "stochastic_sampling": False,
    "time_shift_type": "exponential",
    "use_beta_sigmas": False,
    "use_dynamic_shifting": True,
    "use_exponential_sigmas": False,
    "use_karras_sigmas": False,
}


def pick_dtype(name: str | None = None):
    """bf16 en GPUs que lo tienen nativo (Ampere o posterior: L4, A10G, A100);
    fp16 en las anteriores (T4 de Colab), donde bf16 es emulado y lento y la
    atención eficiente en memoria de PyTorch solo existe en fp16. diffusers
    recorta las activaciones de Qwen-Image en fp16 para que no desborden;
    measure.py comprueba igualmente que no salgan imágenes negras o NaN."""
    import torch

    name = (name or os.environ.get("QWEN_DTYPE", "auto")).lower()
    if name == "auto":
        native = torch.cuda.is_available() and torch.cuda.is_bf16_supported(including_emulation=False)
        return torch.bfloat16 if native else torch.float16
    return torch.float16 if name in ("fp16", "float16") else torch.bfloat16


def load(model: str = MODEL, parts: str = "all", quant: str = "nf4", lightning: bool = True, dtype=None,
         device: str = "cuda", components: dict | None = None):
    """parts: "all" (servidor), "text" (solo codificar) o "image" (solo generar).
    quant: "nf4" (bitsandbytes 4 bits) o "none". components: piezas ya
    construidas (tests con un modelo diminuto)."""
    from diffusers import FlowMatchEulerDiscreteScheduler, QwenImageEditPipeline

    dtype = dtype or pick_dtype()
    kw = dict(components or {})
    need_text, need_image = parts in ("all", "text"), parts in ("all", "image")
    if not need_image:
        kw.setdefault("transformer", None)
        kw.setdefault("vae", None)
    if not need_text:
        kw.setdefault("text_encoder", None)
    if quant == "nf4":
        if need_image and "transformer" not in kw:
            from diffusers import BitsAndBytesConfig, QwenImageTransformer2DModel

            q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype)
            kw["transformer"] = QwenImageTransformer2DModel.from_pretrained(
                model, subfolder="transformer", quantization_config=q, torch_dtype=dtype)
        if need_text and "text_encoder" not in kw:
            from transformers import BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

            q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype)
            kw["text_encoder"] = Qwen2_5_VLForConditionalGeneration.from_pretrained(
                model, subfolder="text_encoder", quantization_config=q, torch_dtype=dtype)
    if lightning and need_image:
        kw["scheduler"] = FlowMatchEulerDiscreteScheduler.from_config(LIGHTNING_SCHEDULER)
    if components is not None:
        pipe = QwenImageEditPipeline(**{k: kw.get(k) for k in
                                        ("scheduler", "vae", "text_encoder", "tokenizer", "processor", "transformer")})
    else:
        pipe = QwenImageEditPipeline.from_pretrained(model, torch_dtype=dtype, **kw)
    if lightning and need_image:
        pipe.load_lora_weights(LIGHTNING_REPO, weight_name=LIGHTNING_FILE)
    # Los módulos en 4 bits ya se cargan en la GPU; el resto se mueve a mano.
    for name in ("transformer", "text_encoder", "vae"):
        mod = getattr(pipe, name, None)
        if mod is not None and not _is_4bit(mod):
            mod.to(device)
    return pipe


def _is_4bit(mod) -> bool:
    return bool(getattr(mod, "is_loaded_in_4bit", False) or getattr(mod, "quantization_method", None))


def prompt_image(pipe, image: Image.Image) -> Image.Image:
    """La foto como la ve el codificador de texto (~1 MP), igual que hace
    QwenImageEditPipeline.__call__ por dentro."""
    from diffusers.pipelines.qwenimage.pipeline_qwenimage_edit import calculate_dimensions

    w, h, _ = calculate_dimensions(1024 * 1024, image.width / image.height)
    return pipe.image_processor.resize(image, h, w)


def encode(pipe, image: Image.Image, prompt: str, negative: str | None = None) -> dict:
    """Fase 1: instrucción + foto -> embeddings (se guardan en CPU)."""
    with _no_grad():
        return _encode(pipe, image, prompt, negative)


def _encode(pipe, image, prompt, negative):
    dev = pipe._execution_device
    pimg = prompt_image(pipe, image)
    pe, pm = pipe.encode_prompt(prompt=prompt, image=pimg, device=dev)
    out = {"pe": pe.cpu(), "pm": None if pm is None else pm.cpu()}
    if negative:
        ne, nm = pipe.encode_prompt(prompt=negative, image=pimg, device=dev)
        out.update(ne=ne.cpu(), nm=None if nm is None else nm.cpu())
    return out


def render(pipe, image: Image.Image, emb: dict, steps: int, true_cfg: float, seed: int) -> Image.Image:
    """Fase 2: embeddings + foto -> imagen editada."""
    with _no_grad():
        return _render(pipe, image, emb, steps, true_cfg, seed)


def _render(pipe, image, emb, steps, true_cfg, seed):
    import torch

    dev = pipe._execution_device
    dtype = pipe.transformer.dtype
    kw = dict(image=image, prompt_embeds=emb["pe"].to(dev, dtype),
              prompt_embeds_mask=None if emb["pm"] is None else emb["pm"].to(dev),
              num_inference_steps=steps, true_cfg_scale=true_cfg,
              generator=torch.Generator(device="cpu").manual_seed(seed))
    if true_cfg > 1 and emb.get("ne") is not None:
        kw.update(negative_prompt_embeds=emb["ne"].to(dev, dtype),
                  negative_prompt_embeds_mask=None if emb["nm"] is None else emb["nm"].to(dev))
    return pipe(**kw).images[0]


class LocalClient:
    """Misma interfaz que InferenceClient.image_to_image (lo que usa app.generate)."""

    def __init__(self, model: str = MODEL, pipe=None):
        self.lightning = os.environ.get("QWEN_LIGHTNING", "0") == "1"
        self.pipe = pipe or load(model, "all", quant=os.environ.get("QWEN_QUANT", "nf4"), lightning=self.lightning)
        self.lock = threading.Lock()  # una GPU: las variantes van en serie

    def image_to_image(self, image: bytes, prompt: str, negative_prompt: str | None = None,
                       num_inference_steps: int = 30, guidance_scale: float = 4.0, model: str | None = None, **_):
        src = Image.open(io.BytesIO(image)).convert("RGB")
        steps, cfg = (8, 1.0) if self.lightning else (num_inference_steps, guidance_scale)
        with self.lock:
            emb = encode(self.pipe, src, prompt, negative_prompt if cfg > 1 else None)
            return render(self.pipe, src, emb, steps, cfg, random.randrange(2**31))


def _no_grad():
    import torch

    return torch.inference_mode()


_client = None
_client_lock = threading.Lock()


def get_client(model: str = MODEL) -> LocalClient:
    global _client
    with _client_lock:
        if _client is None:
            _client = LocalClient(model if not model.startswith("http") else MODEL)
    return _client


In [ ]:
%%writefile measure.py
"""Medición de QA del pipeline Qwen-Image-Edit en Google Colab (GPU gratuita).

Ejecuta el MISMO código que producción (pipeline.py: instrucción, filtro de
similitud antes del puntuador, regeneración de las casi idénticas,
puntuador, elección) pero con el modelo en la GPU de Colab vía diffusers
(local_qwen.py) en lugar de la Inference API de pago.

En una T4 (16 GB) no caben a la vez el codificador (Qwen2.5-VL 7B) y el
transformer (20B), ni en 4 bits, así que va por fases:

1. Codificar todas las instrucciones (+ foto) con Qwen2.5-VL en 4 bits.
2. Liberar la GPU; cargar el transformer en 4 bits (+ LoRA Lightning, 8
   pasos) y generar 2 variantes por foto; filtro de similitud; regenerar
   una vez las casi idénticas.
3. Puntuar (CPU) solo las que pasan el filtro.
4. Opcional: SDXL img2img sobre las mismas fotos, como referencia de calidad
   (SDXL ya no se sirve por la API; aquí corre en la propia GPU).

Salida en out_dir: report.md, data.json y una rejilla por foto
(original | SDXL | variantes Qwen con su similitud y nota).
"""

from __future__ import annotations

import gc
import io
import json
import re
import time
import traceback
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw

import pipeline

UA = "HomeAI-QA/1.0 (https://github.com/JuanCopado/homeai-claude)"
# Las 10 fotos reales de la validación de segmentación (Wikimedia Commons,
# licencias libres; la atribución se escribe en el informe).
PHOTOS = [
    "File:Living room in apartment of Condomínio do Edifício Zaher, Le Blond, Rio de Janeiro, Brazil.jpg",
    "File:Apartment Living Room 4 2018-09-28.jpg",
    "File:Empty apartment living room.jpg",
    "File:A Dingy Apartment Kitchen in Canada.jpg",
    "File:Empty apartment in Berlin with fitted kitchen and chair.jpg",
    "File:Apartment Kitchen 1 2018-09-28.jpg",
    "File:Bedroom in a serviced apartment in London.jpg",
    "File:Bedroom, Interior of apartment in Brisbane, 2025, 02.jpg",
    "File:Bedroom, Interior of apartment in Brisbane, 2025, 01.jpg",
    "File:02023 0113 Bathroom, Presidential private apartment in Wawel Castle.jpg",
]
STYLES = [
    "estilo nórdico escandinavo, paredes blancas, madera clara, textiles de lino, plantas",
    "estilo industrial, ladrillo visto, metal negro, madera oscura",
    "estilo japandi, tonos tierra, madera natural, líneas simples",
    "estilo mediterráneo, blanco y azul, cerámica, fibras naturales",
    "estilo moderno minimalista, gris claro, muebles lacados, iluminación indirecta",
]
# Intensidad por foto: la mayoría la de la interfaz (0,55); tres "retoque"
# (0,3), donde es más probable que el modelo casi no cambie nada y el filtro
# de similitud tenga trabajo, y dos "rediseño completo" (0,75).
STRENGTHS = [0.55, 0.3, 0.55, 0.75, 0.3, 0.55, 0.55, 0.3, 0.75, 0.55]


def default_jobs():
    return [{"n": i + 1, "title": t, "style": STYLES[i % len(STYLES)], "strength": STRENGTHS[i]}
            for i, t in enumerate(PHOTOS)]


def fetch_photo(title: str, width: int = 1024):
    import requests

    r = requests.get("https://commons.wikimedia.org/w/api.php", headers={"User-Agent": UA}, timeout=60, params={
        "action": "query", "titles": title, "prop": "imageinfo", "iiprop": "url|extmetadata",
        "iiurlwidth": width, "format": "json"})
    info = next(iter(r.json()["query"]["pages"].values()))["imageinfo"][0]
    meta = info.get("extmetadata", {})
    artist = re.sub(r"<[^>]+>", "", meta.get("Artist", {}).get("value", "?")).strip()
    credit = f"[{title}]({info['descriptionurl']}) — {artist}, {meta.get('LicenseShortName', {}).get('value', '?')}"
    img = Image.open(io.BytesIO(requests.get(info["thumburl"], headers={"User-Agent": UA}, timeout=120).content))
    return img.convert("RGB"), credit


def prepare(img: Image.Image, max_side: int = 1024) -> Image.Image:
    """Como app.preprocess: lado mayor ≤ 1024 y múltiplo de 8."""
    s = min(1.0, max_side / max(img.size))
    w, h = max(8, int(img.width * s) // 8 * 8), max(8, int(img.height * s) // 8 * 8)
    return img.resize((w, h), Image.LANCZOS)


def broken(img: Image.Image) -> str | None:
    a = np.asarray(img.convert("RGB"), dtype=np.float32)
    if not np.isfinite(a).all():
        return "nan"
    if a.max() < 8 or a.std() < 1.0:
        return "negra_o_plana"  # síntoma típico de desbordamiento en fp16
    return None


def _free():
    gc.collect()
    try:
        import torch

        torch.cuda.empty_cache()
    except Exception:  # noqa: BLE001
        pass


def drop_cache(repo: str, folder: str) -> float:
    """Borra de la caché de HF una subcarpeta ya usada (p. ej. el codificador
    tras la fase 1) para que el disco de Colab (~80 GB libres) no se llene:
    codificador 16,6 GB + transformer 40,9 GB + SDXL 7 GB no caben a la vez
    con holgura. Devuelve los GB liberados."""
    try:
        from huggingface_hub import scan_cache_dir
    except Exception:  # noqa: BLE001
        return 0.0
    freed = 0
    for r in scan_cache_dir().repos:
        if r.repo_id != repo:
            continue
        for rev in r.revisions:
            for f in rev.files:
                if f.file_name and f"/{folder}/" in str(f.file_path).replace("\\", "/"):
                    for p in (f.blob_path, f.file_path):
                        try:
                            freed += p.stat().st_size if p == f.blob_path else 0
                            p.unlink()
                        except OSError:
                            pass
    return round(freed / 2**30, 1)


def _vram():
    try:
        import torch

        return round(torch.cuda.max_memory_allocated() / 2**30, 1) if torch.cuda.is_available() else None
    except Exception:  # noqa: BLE001
        return None


def run(out_dir: str | Path = "qa_qwen", jobs=None, variants: int = 2, lightning: bool = True, quant: str = "nf4",
        max_similarity: float = 0.89, min_score: float = 0.5, sdxl: bool = True, steps: int | None = None,
        loaders: dict | None = None, fetch=fetch_photo, log=print, free_disk: bool = True) -> dict:
    """loaders: {"text": f(), "image": f(), "sdxl": f()} para sustituir la carga
    real (el smoke test de CI usa un modelo diminuto en CPU)."""
    import local_qwen

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    jobs = jobs or default_jobs()
    loaders = loaders or {}
    true_cfg = 1.0 if lightning else 4.0
    steps = steps or (8 if lightning else 30)
    data = {"config": {"variants": variants, "lightning": lightning, "quant": quant, "steps": steps, "true_cfg": true_cfg,
                       "max_similarity": max_similarity, "min_score": min_score}, "jobs": [], "errors": []}

    # 0. Fotos
    for j in jobs:
        img, credit = fetch(j["title"])
        j["image"], j["credit"] = prepare(img), credit
        j["prompt"] = pipeline.build_prompt(j["style"], [], j["strength"], edit=True)
        log(f"foto {j['n']:02d} {j['image'].size}")

    # 1. Codificar (Qwen2.5-VL)
    t0 = time.perf_counter()
    text = loaders.get("text", lambda: local_qwen.load(parts="text", quant=quant, lightning=False))()
    for j in jobs:
        j["emb"] = local_qwen.encode(text, j["image"], j["prompt"], pipeline.DEFAULT_NEGATIVE)
    data["encode_seconds"] = round(time.perf_counter() - t0, 1)
    data["vram_text_gb"] = _vram()
    del text
    _free()
    log(f"codificado en {data['encode_seconds']} s")
    if free_disk:
        log(f"caché del codificador borrada: {drop_cache(local_qwen.MODEL, 'text_encoder')} GB")

    # 2. Generar + filtro de similitud + regenerar las casi idénticas
    t0 = time.perf_counter()
    try:
        editor = loaders.get("image", lambda: local_qwen.load(parts="image", quant=quant, lightning=lightning))()
    except Exception as exc:  # noqa: BLE001 - sin LoRA se puede seguir, más lento
        if not lightning:
            raise
        data["errors"].append(f"LoRA Lightning no cargó ({type(exc).__name__}: {exc}); se sigue sin ella")
        lightning, true_cfg, steps = False, 4.0, 30
        data["config"].update(lightning=False, true_cfg=true_cfg, steps=steps)
        _free()
        editor = local_qwen.load(parts="image", quant=quant, lightning=False)
    data["load_image_seconds"] = round(time.perf_counter() - t0, 1)
    seed = 1000

    def gen(j):
        nonlocal seed
        seed += 1
        t = time.perf_counter()
        try:
            img = local_qwen.render(editor, j["image"], j["emb"], steps, true_cfg, seed)
        except Exception as exc:  # noqa: BLE001
            data["errors"].append(f"foto {j['n']:02d}: {type(exc).__name__}: {exc}")
            log(traceback.format_exc())
            _free()
            return None
        img = img.convert("RGB").resize(j["image"].size, Image.LANCZOS)
        v = {"image": img, "seconds": round(time.perf_counter() - t, 1), "seed": seed, "broken": broken(img)}
        log(f"  foto {j['n']:02d} semilla {seed}: {v['seconds']} s" + (f" ⚠️ {v['broken']}" if v["broken"] else ""))
        return v

    for j in jobs:
        vs = [v for v in (gen(j) for _ in range(variants)) if v]
        pipeline.check_similarity(vs, j["image"], None, max_similarity)
        same = [v for v in vs if v.get("reason") == "sin_cambios"]
        more = [v for v in (gen(j) for _ in same) if v]
        pipeline.check_similarity(more, j["image"], None, max_similarity)
        for v in more:
            v["regenerated"] = True
        j["variants"] = vs + more
    data["generate_seconds"] = round(time.perf_counter() - t0, 1)
    data["vram_image_gb"] = _vram()
    editor = None  # noqa: F841 - libera la GPU antes de SDXL
    _free()
    if free_disk and sdxl:
        log(f"caché del transformer borrada: {drop_cache(local_qwen.MODEL, 'transformer')} GB")

    # 3. Puntuador, solo sobre las que pasaron el filtro
    for j in jobs:
        pipeline.score_variants(j["variants"], min_score)
        if j["variants"]:
            j["best"] = pipeline.pick_best(j["variants"])

    # 4. Referencia SDXL (misma foto, prompt y strength del pipeline antiguo)
    if sdxl:
        try:
            ref = loaders.get("sdxl", _load_sdxl)()
            import quality

            for j in jobs:
                t = time.perf_counter()
                img = ref(prompt=pipeline.build_prompt(j["style"], [], edit=False), negative_prompt=pipeline.DEFAULT_NEGATIVE,
                          image=j["image"], strength=j["strength"], num_inference_steps=30, guidance_scale=7.0).images[0]
                img = img.convert("RGB").resize(j["image"].size, Image.LANCZOS)
                j["sdxl"] = {"image": img, "seconds": round(time.perf_counter() - t, 1),
                             "similarity": quality.similarity(img, j["image"]), **quality.score(img)}
            del ref
            _free()
        except Exception as exc:  # noqa: BLE001
            data["errors"].append(f"SDXL: {type(exc).__name__}: {exc}")

    _write(out, jobs, data)
    data["jobs"] = jobs
    return data


def _load_sdxl():
    import torch
    from diffusers import StableDiffusionXLImg2ImgPipeline

    return StableDiffusionXLImg2ImgPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, variant="fp16").to("cuda")


def _tile(img, label, h=300):
    t = img.copy()
    t.thumbnail((h * 2, h))
    d = ImageDraw.Draw(t)
    d.rectangle((0, 0, 8 + 6 * len(label), 16), fill=(255, 255, 255))
    d.text((4, 2), label, fill=(0, 0, 0))
    return t


def _write(out: Path, jobs, data):
    gens = [v for j in jobs for v in j.get("variants", [])]
    first = [v for v in gens if not v.get("regenerated")]
    regen = [v for v in gens if v.get("regenerated")]
    near = [v for v in first if v.get("reason") == "sin_cambios"]
    secs = [v["seconds"] for v in gens]
    lines = ["# Qwen-Image-Edit en Colab: medición del pipeline", "",
             f"Configuración: `{json.dumps(data['config'], ensure_ascii=False)}`", "",
             "## Estabilidad", "",
             f"- Generaciones completadas: **{len(gens)}** (primeras {len(first)} + regeneradas {len(regen)}); "
             f"errores: **{len(data['errors'])}**.",
             f"- Imágenes negras/NaN: **{sum(1 for v in gens if v.get('broken'))}**.",
             (f"- Tiempo por generación: media {np.mean(secs):.1f} s, mín {min(secs):.1f}, máx {max(secs):.1f}."
              if secs else "- Sin tiempos."),
             f"- Codificación de todas las instrucciones: {data.get('encode_seconds')} s; carga del transformer: "
             f"{data.get('load_image_seconds')} s.",
             f"- VRAM máxima: codificador {data.get('vram_text_gb')} GB, transformer {data.get('vram_image_gb')} GB.", ""]
    if data["errors"]:
        lines += ["Errores:", "", *[f"- `{e}`" for e in data["errors"]], ""]
    lines += ["## Filtro de similitud (antes del puntuador)", "",
              f"- Umbral: similitud CLIP > {data['config']['max_similarity']} → casi idéntica.",
              f"- Descartadas en la primera tanda: **{len(near)}/{len(first)}**"
              + (f" ({100 * len(near) / len(first):.0f} %)." if first else "."),
              f"- Regeneradas: {len(regen)}; de ellas, otra vez casi idénticas: "
              f"{sum(1 for v in regen if v.get('reason') == 'sin_cambios')}.",
              "- Similitudes de todas las generaciones: "
              + ", ".join(str(v.get("similarity")) for v in gens), ""]
    lines += ["## Calidad", "",
              "| Foto | Intensidad | Estilo | SDXL (nota / similitud) | Qwen: nota / similitud por variante | Elegida |",
              "|---|---|---|---|---|---|"]
    for j in jobs:
        sd = j.get("sdxl")
        sd_txt = f"{sd['score']} / {sd['similarity']}" if sd else "—"
        vs = "; ".join(f"v{i + 1}{'ʳ' if v.get('regenerated') else ''} {v.get('score', '—')} / {v.get('similarity')}"
                       + (f" ({v['reason']})" if v.get("reason") else "") for i, v in enumerate(j.get("variants", [])))
        best = j.get("best")
        lines.append(f"| {j['n']:02d} | {j['strength']} | {j['style'].split(',')[0]} | {sd_txt} | {vs or '—'} | "
                     f"{'v' + str(best + 1) if best is not None else '—'} |")
        tiles = [_tile(j["image"], "original")]
        if sd:
            tiles.append(_tile(sd["image"], f"SDXL s={sd['score']} sim={sd['similarity']}"))
        for i, v in enumerate(j.get("variants", [])):
            mark = "*" if i == best else ""
            tiles.append(_tile(v["image"], f"Qwen v{i + 1}{mark} s={v.get('score', '-')} sim={v.get('similarity')}"
                               + (" DESCARTADA" if v.get("discarded") else "")))
        grid = Image.new("RGB", (sum(t.width for t in tiles), max(t.height for t in tiles)), "white")
        x = 0
        for t in tiles:
            grid.paste(t, (x, 0))
            x += t.width
        grid.save(out / f"{j['n']:02d}.jpg", quality=85)
    q = [v["score"] for v in gens if v.get("score") is not None]
    s = [j["sdxl"]["score"] for j in jobs if j.get("sdxl")]
    lines += ["", f"Nota media Qwen (las que pasaron el filtro): {np.mean(q):.3f}" if q else "",
              f"Nota media SDXL: {np.mean(s):.3f}" if s else "",
              "", "La comparación subjetiva se hace mirando las rejillas:", "",
              *[f"![{j['n']:02d}]({j['n']:02d}.jpg)" for j in jobs], "", "## Fotos", "",
              *[f"- {j['n']:02d}: {j.get('credit', j['title'])}" for j in jobs]]
    (out / "report.md").write_text("\n".join(lines), encoding="utf-8")
    slim = {k: v for k, v in data.items() if k != "jobs"}
    slim["jobs"] = [{"n": j["n"], "title": j["title"], "style": j["style"], "strength": j["strength"],
                     "prompt": j["prompt"], "best": j.get("best"),
                     "sdxl": {k: v for k, v in j["sdxl"].items() if k != "image"} if j.get("sdxl") else None,
                     "variants": [{k: v for k, v in x.items() if k != "image"} for x in j.get("variants", [])]}
                    for j in jobs]
    (out / "data.json").write_text(json.dumps(slim, ensure_ascii=False, indent=1), encoding="utf-8")


## Configuración

In [ ]:
import os
VARIANTS = 2        # variantes por foto (el máximo del servidor)
LIGHTNING = True    # LoRA Lightning: 8 pasos sin CFG. False = 30 pasos con CFG (unas 8 veces más lento)
SDXL = True         # referencia SDXL sobre las mismas fotos
os.environ["QWEN_DTYPE"] = "auto"   # auto: fp16 en T4, bf16 en L4/A100. Prueba "bf16" si salen imágenes negras.
os.environ["TOKENIZERS_PARALLELISM"] = "false"


## Medición

In [ ]:
import measure
data = measure.run("qa_qwen", variants=VARIANTS, lightning=LIGHTNING, sdxl=SDXL)

## Informe y descarga

In [ ]:
from IPython.display import Markdown, display
import shutil
display(Markdown(open("qa_qwen/report.md", encoding="utf-8").read().split("La comparación subjetiva")[0]))
shutil.make_archive("qa_qwen", "zip", "qa_qwen")
try:
    from google.colab import files
    files.download("qa_qwen.zip")
except ImportError:
    print("Resultados en qa_qwen.zip")